# Análise Sentimental de Letras de Músicas Brasileiras por Gênero

Este notebook realiza a coleta de letras de músicas brasileiras de diferentes gêneros
usando a API do **MusicBrainz** para buscar artistas e o site **letras.com.br** para
extrair as letras.

**Gêneros cobertos:** MPB, Rock, Sertanejo, Funk, Rap  
**Artistas:** 50 (10 por gênero)  
**Músicas:** 2 por artista = 100 músicas no total

## 1. Instalação de Dependências

In [1]:
!pip install requests beautifulsoup4 pandas

## 2. Imports

In [2]:
import re
import time
import unicodedata

import pandas as pd
import requests
from bs4 import BeautifulSoup
from requests.adapters import HTTPAdapter
from requests.exceptions import RequestException, SSLError
from urllib.parse import urljoin, urlparse
from urllib3.util.retry import Retry

## 3. Configurações e Constantes

In [3]:
# URLs base
MB_URL   = "https://musicbrainz.org/ws/2/artist/"
BASE_URL = "https://www.letras.com.br/"

# Headers para a API do MusicBrainz
HEADERS_MB = {
    "User-Agent": "NLPMusicProject/1.0 (anadognini15@exemplo.com)",
    "Accept": "application/json"
}

# Headers para o letras.com.br — User-Agent de browser real para evitar bloqueio
HEADERS_SITE = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Accept-Language": "pt-BR,pt;q=0.9",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Referer": "https://www.letras.com.br/",
}

# Gêneros a coletar
GENRES = ["mpb", "rock", "sertanejo", "funk", "rap"]

# Exceções manuais: artistas cujo slug no letras.com.br difere do gerado automaticamente
slug_exceptions = {
    "Racionais MC\u2019s": "racionais-mcs",   # apóstrofo curvo '
    "Racionais MC's": "racionais-mcs",   # apóstrofo reto '
    "Maria Bethânia": "maria-bethania",
    "Zezé Di Camargo": "zeze-di-camargo",
    "Chitãozinho & Xororó": "chitaozinho-e-xororo",
    "Milionário & José Rico": "milionario-e-jose-rico",
    "Claudinho & Buchecha": "claudinho-e-buchecha",
}

## 4. Funções Auxiliares

In [4]:
# Cria uma sessão HTTP com retry automático para erros de rede

def create_session() -> requests.Session:
    session = requests.Session()

    retry = Retry(
        total=5,
        connect=5,
        read=5,
        backoff_factor=2,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"]
    )

    adapter = HTTPAdapter(max_retries=retry)
    session.mount("https://", adapter)
    session.mount("http://",  adapter)

    return session

session_mb = create_session()

In [5]:
# Faz uma requisição GET e retorna um objeto BeautifulSoup

def get_soup(url: str, headers: dict = None, sleep_seconds: float = 2.0) -> BeautifulSoup:
    if headers is None:
        headers = HEADERS_SITE

    time.sleep(sleep_seconds)

    resp = requests.get(url, headers=headers, timeout=30)
    resp.raise_for_status()

    return BeautifulSoup(resp.text, "html.parser")

In [6]:
# Converte um nome de artista para o formato de slug usado no letras.com.br

def slugify_artist_name(name: str) -> str:
    name = name.lower().strip()
    name = unicodedata.normalize("NFKD", name)
    name = "".join(c for c in name if not unicodedata.combining(c))
    name = name.replace("&", " e ")
    name = re.sub(r"[''`\u00b4]", "", name)
    name = re.sub(r"[^a-z0-9\s-]", " ", name)
    name = re.sub(r"\s+", "-", name).strip("-")
    name = re.sub(r"-{2,}", "-", name)

    return name

# Monta a URL da página do artista no letras.com.br

def build_artist_url(artist_slug: str) -> str:
    return urljoin(BASE_URL, f"{artist_slug}/")

In [7]:
# Remove ruídos comuns das letras coletadas

def clean_lyrics(text: str) -> str:
    text = text.replace("\xa0", " ")
    text = re.sub(r"\r", "\n", text)
    text = re.sub(r"\[.*?\]", "", text)               # remove tags [refrão], [verso], etc.
    text = re.sub(r"\(.*?x\)", "", text, flags=re.IGNORECASE)  # remove repetições (2x)
    text = re.sub(r"[ \t]+\n", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()

## 5. Coleta de Artistas via MusicBrainz

In [8]:
# Busca artistas brasileiros por gênero na API do MusicBrainz

def search_artists_by_genre(genre: str, limit: int = 15, country: str = "BR") -> list:
    params = {
        "query": f'tag:"{genre}" AND country:{country}',
        "fmt":   "json",
        "limit": limit
    }

    try:
        resp = session_mb.get(MB_URL, params=params, headers=HEADERS_MB, timeout=(10, 30))
        resp.raise_for_status()
        return resp.json().get("artists", [])
    except SSLError as e:
        print(f"Erro SSL ao buscar gênero {genre}: {e}")
        return []
    except RequestException as e:
        print(f"Erro de requisição ao buscar gênero {genre}: {e}")

        return []

In [9]:
rows = []

for genre in GENRES:
    artists = search_artists_by_genre(genre=genre, limit=15)

    if not artists:
        print(f"Nenhum artista retornado para {genre}")
        continue

    for artist in artists:
        rows.append({
            "genre_seed":   genre,
            "artist_name":  artist.get("name"),
            "artist_id_mb": artist.get("id"),
            "country":      artist.get("country"),
            "score":        artist.get("score")
        })

    time.sleep(1.2)  # respeita o rate limit do MusicBrainz

In [10]:
# df_artists_all: todos os artistas retornados, ordenados por score,
# sem limitar a 10 por gênero ainda — filtrado na etapa seguinte

df_artists_all = (
    pd.DataFrame(rows)
    .sort_values(["genre_seed", "score"], ascending=[True, False])
    .drop_duplicates(subset=["genre_seed", "artist_name"])
    .reset_index(drop=True)
)

print(f"Total de artistas (antes da seleção final): {len(df_artists_all)}")
print(df_artists_all.groupby("genre_seed")["artist_name"].count().to_string())

Total de artistas (antes da seleção final): 75
genre_seed
funk         15
mpb          15
rap          15
rock         15
sertanejo    15


In [11]:
# Artistas excluídos por gênero:
#   rock  → Gilberto Gil e Jorge Ben Jor aparecem duplicados (também em mpb),
#            seriam contados duas vezes no dataset
#   funk  → DJ Marky não possui página no letras.com.br

excluded_artists = {
    "funk": {"DJ Marky"},
    "rock": {"Gilberto Gil", "Jorge Ben Jor"},
}

# Seleciona os top `n_per_genre` artistas por gênero, excluindo artistas
# definidos em EXCLUDED_ARTISTS (duplicados entre gêneros ou sem página
# no letras.com.br)

def select_final_artists(df_all: pd.DataFrame, n_per_genre: int = 10) -> pd.DataFrame:
    selected = []
    for genre in GENRES:
        genre_df = df_all[df_all["genre_seed"] == genre].copy()
        to_exclude = excluded_artists.get(genre, set())
        genre_df = genre_df[~genre_df["artist_name"].isin(to_exclude)]
        selected.append(genre_df.head(n_per_genre))
    return pd.concat(selected, ignore_index=True)

In [12]:
df_artists_final = select_final_artists(df_artists_all, n_per_genre=10)

df_artists_final["artist_slug"] = df_artists_final["artist_name"].apply(
    lambda x: slug_exceptions.get(x, slugify_artist_name(x))
)

df_artists_final.to_csv("artists_final.csv", index=False, encoding="utf-8")

print(f"Total de artistas selecionados: {len(df_artists_final)}")
print(df_artists_final.groupby("genre_seed").size().to_string())
display(df_artists_final[["genre_seed", "artist_name", "artist_slug"]])

Total de artistas selecionados: 50
genre_seed
funk         10
mpb          10
rap          10
rock         10
sertanejo    10


,genre_seed,artist_name,artist_slug
0,mpb,Maria Bethânia,maria-bethania
1,mpb,Fagner,fagner
2,mpb,Marcos Valle,marcos-valle
3,mpb,Ney Matogrosso,ney-matogrosso
4,mpb,Elis Regina,elis-regina
5,mpb,Gonzaguinha,gonzaguinha
6,mpb,Gilberto Gil,gilberto-gil
7,mpb,Alcione,alcione
8,mpb,Simone,simone
9,mpb,Jorge Ben Jor,jorge-ben-jor


## 6. Extração de Músicas e Letras do letras.com.br

In [13]:
# Extrai os links de músicas da página de um artista no letras.com.br
# Seletor baseado na estrutura real confirmada via DevTools:
# ul.list-column-2 > li > a.item-box.item-list

def extract_song_links_from_artist_page(artist_url: str, artist_slug: str) -> list:
    soup = get_soup(artist_url, headers=HEADERS_SITE, sleep_seconds=2.0)

    song_links = []
    seen = set()

    invalid_slugs = {
        "", "discografia", "biografia", "curiosidades",
        "traducao", "top-musicas", "videos", "fotos"
    }

    for a in soup.select("a.item-box.item-list, ul.list-column-2 li a.item-box"):
        href = a.get("href", "").strip()
        if not href:
            continue

        # Remove âncora (#top=...) e query string ANTES do urlparse
        href_clean = href.split("#")[0].split("?")[0]
        full_url   = urljoin(BASE_URL, href_clean).rstrip("/")

        path  = urlparse(full_url).path.strip("/")
        parts = path.split("/")

        if len(parts) != 2:
            continue
        if parts[0] != artist_slug:
            continue
        if parts[1] in invalid_slugs:
            continue
        if full_url in seen:
            continue

        seen.add(full_url)

        span  = a.find("span", class_="title-default")
        title = span.get_text(strip=True) if span else parts[1].replace("-", " ").title()

        song_links.append({"song_title_hint": title, "song_url": full_url})

    return song_links

In [14]:
# Extrai o título e a letra de uma música no letras.com.br.
# Container da letra confirmado via DevTools: section.lyrics-section

def extract_lyrics_from_song_page(song_url: str) -> dict:
    soup = get_soup(song_url, headers=HEADERS_SITE, sleep_seconds=2.0)

    # Título: tenta seletores específicos antes do <h1> genérico do site
    title = "Sem título"
    for selector in ["h1.title-intro", "h1.cnt-head-title", ".content-title-section h1", "h2"]:
        tag = soup.select_one(selector)
        if tag:
            candidate = tag.get_text(" ", strip=True)
            if "letras.com" not in candidate.lower():
                title = candidate
                break

    # Fallback: deriva o título do slug na URL
    if title == "Sem título" or "letras.com" in title.lower():
        title = song_url.rstrip("/").split("/")[-1].replace("-", " ").title()

    # Letra: container confirmado via DevTools
    lyric_section = soup.find("section", class_="lyrics-section")

    if lyric_section is None:
        return {"song_url": song_url, "song_title": title, "lyrics_raw": "", "lyrics_clean": ""}

    strophes = []
    for p in lyric_section.find_all("p"):
        for br in p.find_all("br"):
            br.replace_with("\n")
        text = p.get_text("\n", strip=True)
        if text:
            strophes.append(text)

    lyrics_raw   = "\n\n".join(strophes)
    lyrics_clean = clean_lyrics(lyrics_raw)

    return {
        "song_url":    song_url,
        "song_title":  title,
        "lyrics_raw":  lyrics_raw,
        "lyrics_clean": lyrics_clean
    }

In [15]:
# Percorre o DataFrame de artistas e coleta `songs_per_artist` letras para cada um

def collect_songs_from_artists(df_artists: pd.DataFrame, songs_per_artist: int = 2) -> pd.DataFrame:
    rows  = []
    total = len(df_artists)

    for i, (_, artist_row) in enumerate(df_artists.iterrows(), 1):
        artist_name = artist_row["artist_name"]
        genre       = artist_row["genre_seed"]
        artist_slug = artist_row["artist_slug"]
        artist_url  = build_artist_url(artist_slug)

        print(f"[{i}/{total}] Coletando artista: {artist_name} | gênero: {genre}")

        try:
            song_links = extract_song_links_from_artist_page(artist_url, artist_slug)

            if not song_links:
                print(f"   ✗ Nenhuma música encontrada")
                continue

            collected = 0

            for song in song_links[:20]:  # tenta até 20 links para achar songs_per_artist válidas
                if collected >= songs_per_artist:
                    break
                try:
                    song_data = extract_lyrics_from_song_page(song["song_url"])

                    if not song_data["lyrics_clean"]:
                        continue
                    if len(song_data["lyrics_clean"].split()) < 30:
                        continue

                    n_words = len(song_data["lyrics_clean"].split())
                    rows.append({
                        "genre":       genre,
                        "artist":      artist_name,
                        "artist_slug": artist_slug,
                        "artist_url":  artist_url,
                        "song_title":  song_data["song_title"],
                        "song_url":    song_data["song_url"],
                        "lyrics_raw":  song_data["lyrics_raw"],
                        "lyrics_clean": song_data["lyrics_clean"],
                        "n_words":     n_words
                    })
                    collected += 1
                    print(f"   ✓ {song_data['song_title']}")

                except Exception as e:
                    print(f"   ✗ Erro em {song['song_url']}: {e}")

            if collected == 0:
                print(f"   ✗ Nenhuma letra válida coletada")
            print()

        except Exception as e:
            print(f"   ✗ Erro ao processar artista: {e}\n")

    df_songs = pd.DataFrame(rows, columns=[
        "genre", "artist", "artist_slug", "artist_url",
        "song_title", "song_url", "lyrics_raw", "lyrics_clean", "n_words"
    ])

    if not df_songs.empty:
        df_songs["n_chars"] = df_songs["lyrics_clean"].fillna("").str.len()
        df_songs = df_songs.drop_duplicates(subset=["artist", "song_title"]).reset_index(drop=True)

    return df_songs

## 7. Execução da Coleta

In [16]:
df_songs = collect_songs_from_artists(df_artists_final, songs_per_artist=2)

print(f"\nTotal de músicas coletadas: {len(df_songs)}")
print(df_songs.groupby("genre")[["song_title"]].count().rename(columns={"song_title": "músicas"}).to_string())

[1/50] Coletando artista: Maria Bethânia | gênero: mpb
   ✓ Iemanjá Rainha Do Mar
   ✓ Samba Da Bênção

[2/50] Coletando artista: Fagner | gênero: mpb
   ✓ Borbulhas de Amor
   ✓ Oração de São Francisco

[3/50] Coletando artista: Marcos Valle | gênero: mpb
   ✓ Um Novo Tempo
   ✓ Só Se Morre Uma Vez

[4/50] Coletando artista: Ney Matogrosso | gênero: mpb
   ✓ Demarcação Já!
   ✓ Rosa de Hiroshima

[5/50] Coletando artista: Elis Regina | gênero: mpb
   ✓ Preciso Aprender a Ser Só
   ✓ Como Nossos Pais

[6/50] Coletando artista: Gonzaguinha | gênero: mpb
   ✓ O Que É, O Que É
   ✓ Pequena Memória Para Um Tempo Sem Memória

[7/50] Coletando artista: Gilberto Gil | gênero: mpb
   ✓ Cores Vivas
   ✓ O Beija-flor

[8/50] Coletando artista: Alcione | gênero: mpb
   ✓ Coração De Porcelana
   ✓ Trocando Em Miúdos

[9/50] Coletando artista: Simone | gênero: mpb
   ✓ Então É Natal (Happy X-Mas) (War Is Over)
   ✓ Bate O Sino

[10/50] Coletando artista: Jorge Ben Jor | gênero: mpb
   ✓ O Nome do R

## 8. Inspeção e Salvamento dos Dados

In [18]:
print(f"Shape: {df_songs.shape}")

df_songs[["genre", "artist", "song_title", "n_words", "n_chars"]].head(10)

Shape: (100, 10)


,genre,artist,song_title,n_words,n_chars
0,mpb,Maria Bethânia,Iemanjá Rainha Do Mar,163,852
1,mpb,Maria Bethânia,Samba Da Bênção,149,712
2,mpb,Fagner,Borbulhas de Amor,229,1234
3,mpb,Fagner,Oração de São Francisco,109,584
4,mpb,Marcos Valle,Um Novo Tempo,63,297
5,mpb,Marcos Valle,Só Se Morre Uma Vez,100,435
6,mpb,Ney Matogrosso,Demarcação Já!,781,4414
7,mpb,Ney Matogrosso,Rosa de Hiroshima,58,341
8,mpb,Elis Regina,Preciso Aprender a Ser Só,169,800
9,mpb,Elis Regina,Como Nossos Pais,324,1613


In [19]:
df_songs.to_csv("songs.csv", index=False, encoding="utf-8", lineterminator="\n")

print("Arquivo songs.csv salvo com sucesso.")

Arquivo songs.csv salvo com sucesso.


In [20]:
amostra = df_songs[df_songs["artist"] == "Maria Bethânia"].iloc[0]

print(repr(amostra["song_title"]))
print(repr(amostra["lyrics_clean"][:100]))

'Iemanjá Rainha Do Mar'
'Quanto nome tem a Rainha do Mar?\nQuanto nome tem a Rainha do Mar?\nDandalunda, Janaína,\nMarabô, Princ'


In [21]:
df = pd.read_csv('/content/songs.csv')

In [31]:
print(df.loc[99, "lyrics_clean"])

Léo:
Eu tento fazer com que não me percebam aqui
A sombra no rosto que diz: "Não tente invadir"
Respiração ofegante, faço essa porra subir
Sinto o peso nas costas querendo me sucumbir

Insisto no ferro, no meu modo de agir
Cada dia que passa é mais um que quero sumir
Já se olhou no espelho e não conseguia se ver?
Pois é rotina de quem não teme morrer

Levanto e faço de novo com todo prazer
Já tive de tudo, eu sei, mesmo sem merecer
Dedico meu tempo somente pra mim
Sou um anjo gelado, caçador de querubim

Não estranhe se eu passar de cara amarrada
Não me dou bem com as pessoas e não faço fachada
Protagonizo um filme solo sobre o mal e o bem
Mas sempre de joelhos, orando também

Blindão, blin blindão, blindão, blindão
Blindão, blindão, blindão, tem que ser blindão
Blindão, blin blindão, blindão, blindão
Blindão, blindão, blindão, tem que ser blindão

Mais um mês se acabando e o dinheiro indo junto
O aluguel apertando, mais um brinde com o mundo
Blindão, um gole, é o ferro, não corre
Não 